# Hybrid RAG Theory Notebook

This notebook is a theory-first guide to Retrieval-Augmented Generation (RAG) with emphasis on **Hybrid RAG**, **BM25**, **Reciprocal Rank Fusion (RRF)**, and **evaluation design**.

## What this notebook covers
- What RAG is and why it works
- Main types of RAG systems
- Why Hybrid RAG combines sparse and dense retrieval
- BM25 formula and intuition
- RRF formula and fusion logic
- Retrieval evaluation: correct fetch and correct ranking
- Generation evaluation: which metrics matter for which use case

## Mental model
RAG is not only a generation problem. It is a **pipeline quality** problem:

1. Retrieve the right evidence
2. Rank the evidence correctly
3. Build a useful prompt context
4. Generate a grounded answer
5. Evaluate each stage separately

# 1. What Is RAG?

RAG stands for **Retrieval-Augmented Generation**. Instead of asking the LLM to answer from parameters alone, we first retrieve external context and then generate from that context.

A compact formulation is:

$$
\hat{a} = \operatorname{LLM}(q, C_k(q, D))
$$

Where:
- $q$ is the user query
- $D$ is the document collection
- $C_k(q, D)$ is the top-$k$ retrieved context
- $\hat{a}$ is the generated answer

## Why RAG helps
- Reduces hallucination by grounding the answer in retrieved evidence
- Makes answers fresher than pure parametric knowledge
- Lets you adapt to private domain data without full model retraining
- Makes evaluation easier because retrieval and generation can be measured separately

## Core pipeline
```text
User Query -> Retriever -> Top-k Context -> Prompt Builder -> LLM -> Answer
```

# 2. Main Types of RAG

| RAG type | Retrieval signal | Strengths | Weaknesses | Best fit |
| --- | --- | --- | --- | --- |
| Naive RAG | Basic chunk retrieval | Simple to implement | Often weak ranking and chunk quality | Prototypes |
| Sparse RAG | Lexical match such as BM25 | Great for keywords, IDs, codes, exact phrases | Can miss semantic paraphrases | Logs, schemas, product codes, runbooks |
| Dense RAG | Vector similarity from embeddings | Good for semantic similarity | Can miss rare identifiers and exact terms | FAQ, conceptual search, semantic knowledge lookup |
| Hybrid RAG | Sparse + dense combined | Strong balance of exact and semantic retrieval | More moving parts | Enterprise search, analytics, support, knowledge assistants |
| Multi-hop / Graph RAG | Structured reasoning across linked evidence | Better for connected evidence and dependency chains | Higher complexity | Research, compliance, root-cause analysis |
| Agentic / Tool RAG | Retrieval plus tools, APIs, planning | Strong for tasks and workflows | Harder to control and evaluate | Operational copilots, automation agents |
| Corrective / Adaptive RAG | Query rewrite, retry, rerank, self-correction | Better robustness | More latency and orchestration | High-value QA and production assistants |

## Quick intuition
- **Sparse retrieval** answers: "Do the same words appear?"
- **Dense retrieval** answers: "Do these texts mean similar things?"
- **Hybrid retrieval** answers both questions together

# 3. Hybrid RAG Architecture

Hybrid RAG retrieves from **two signals** and then fuses the ranked lists.

$$
C_k^{hybrid}(q) = \operatorname{Fuse}(C^{bm25}(q), C^{dense}(q))
$$

## Diagram
```mermaid
flowchart LR
    Q[User Query] --> N[Normalize / rewrite query]
    N --> S[BM25 sparse retrieval]
    N --> D[Dense vector retrieval]
    S --> F[RRF or weighted fusion]
    D --> F
    F --> R[Optional reranker]
    R --> C[Context builder]
    C --> L[LLM answer]
    L --> E[Retrieval and generation evaluation]
```

If Mermaid rendering is not enabled, use this ASCII view:

```text
                +-------------------+
User Query ---> | Normalize / Rewrite |
                +----------+--------+
                           |
             +-------------+-------------+
             |                           |
             v                           v
      +-------------+             +-------------+
      | BM25 Search |             | Dense Search|
      +------+------+             +------+------+
             |                           |
             +-------------+-------------+
                           v
                    +-------------+
                    |  Fusion     |
                    |  (RRF)      |
                    +------+------+
                           |
                           v
                    +-------------+
                    | Context     |
                    | Builder     |
                    +------+------+
                           |
                           v
                    +-------------+
                    |    LLM      |
                    +-------------+
```

## Why Hybrid RAG often wins
- BM25 catches exact identifiers like `job_7841`, `AR_90_PLUS`, table names, ticket IDs, and version strings
- Dense retrieval catches paraphrases and conceptual similarity
- Fusion reduces the risk of either method failing alone

# 4. BM25 Theory

BM25 is the most common sparse retrieval scoring function. It rewards terms that are:
- frequent in the candidate document
- rare across the corpus
- found in documents that are not excessively long

## BM25 formula

$$
\operatorname{BM25}(q, d) = \sum_{t \in q} IDF(t) \cdot \frac{f(t,d)(k_1 + 1)}{f(t,d) + k_1\left(1 - b + b\cdot\frac{|d|}{avgdl}\right)}
$$

A common IDF form is:

$$
IDF(t) = \log\left(\frac{N - n(t) + 0.5}{n(t) + 0.5}\right)
$$

## Meaning of each symbol
- $q$: query
- $d$: document or chunk
- $t$: a query term
- $f(t,d)$: how many times term $t$ appears in document $d$
- $|d|$: length of document $d$
- $avgdl$: average document length in the corpus
- $N$: number of documents
- $n(t)$: number of documents containing term $t$
- $k_1$: controls term-frequency saturation, often around 1.2 to 2.0
- $b$: controls length normalization, often around 0.75

## Intuition
- If a rare term appears in a document, BM25 gives a bigger reward
- Repeating the same word helps, but with diminishing returns
- Very long documents are normalized so they are not unfairly rewarded

## When BM25 is especially strong
- Schema fields
- Job IDs
- Error codes
- Table names
- Product SKUs
- Legal phrases that should match exactly

# 5. Reciprocal Rank Fusion (RRF)

RRF is a robust way to merge multiple ranked lists without depending on raw score scales being comparable.

## RRF formula

$$
\operatorname{RRF}(d) = \sum_{i=1}^{m} \frac{1}{k + r_i(d)}
$$

Where:
- $m$ is the number of ranked lists
- $r_i(d)$ is the rank of document $d$ in list $i$
- $k$ is a smoothing constant, often 60

## Why RRF is popular
- It uses rank, not raw score magnitude
- It is simple and stable in practice
- It avoids tricky score normalization between BM25 and vector similarity

## Example
Assume one chunk is rank 1 in BM25 and rank 4 in dense retrieval. If $k = 60$:

$$
\operatorname{RRF}(d) = \frac{1}{60 + 1} + \frac{1}{60 + 4}
$$

That chunk gets credit from both systems. Another chunk that appears only in one list gets less combined support.

## Practical takeaway
Use RRF when you want a good default fusion strategy for Hybrid RAG before adding learned rerankers.

# 6. Evaluation Layers in RAG

RAG evaluation should be separated into **retrieval** and **generation**. Otherwise you cannot tell whether failure came from search or from the LLM.

## Layer 1: Correct fetch
Question: **Did the system retrieve the right evidence at all?**

Typical metrics:
- Hit@k
- Recall@k
- Context recall
- Gold-source coverage

## Layer 2: Correct ranking
Question: **Did the system put the best evidence near the top?**

Typical metrics:
- Top-1 accuracy
- MRR
- NDCG@k
- MAP

## Layer 3: Generation quality
Question: **Given the retrieved context, did the LLM answer correctly and stay grounded?**

Typical metrics:
- Exact match or token F1
- ROUGE or BLEU
- Faithfulness / groundedness
- Answer relevance
- Completeness
- Hallucination rate
- Citation precision and citation recall

## Failure diagnosis
- Low fetch metrics: retrieval missed the evidence
- Good fetch but low ranking metrics: evidence exists but is buried too low
- Good retrieval but weak generation metrics: prompt, context construction, or model behavior is the bottleneck

# 7. Retrieval Metrics: Correct Fetch and Correct Ranking

## Correct fetch metrics

### Hit@k
Did at least one relevant item appear in the top-$k$?

$$
Hit@k = \mathbb{1}[\exists\, \text{relevant item in top-}k]
$$

### Precision@k
How much of the top-$k$ is relevant?

$$
Precision@k = \frac{\#(\text{relevant items in top-}k)}{k}
$$

If you average this over many queries, you get mean precision at $k$:

$$
MP@k = \frac{1}{|Q|} \sum_{i=1}^{|Q|} Precision_i@k
$$

Theory: MP@k measures the average purity of the first $k$ results across queries. It is useful when you care about how much noise appears in the retrieved set, but it does not care whether the relevant items are at rank 1 or rank 5 as long as they stay inside the top-$k$.

This is different from MAP, which averages precision values at each relevant hit position.

### Recall@k
How much of all relevant evidence was recovered?

$$
Recall@k = \frac{\#(\text{relevant items in top-}k)}{\#(\text{all relevant items})}
$$

### Coverage / all-gold-fetched
This is useful when a query needs multiple supporting sources.

$$
Coverage = \mathbb{1}[\text{all required sources are present}]
$$

## Correct ranking metrics

### Top-1 accuracy
Is the first retrieved item the one you wanted most?

### MRR
Mean Reciprocal Rank rewards systems that place the first relevant result very high.

$$
MRR = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{rank_i}
$$

Theory: MRR is a first-hit metric. It only looks at the first relevant result and ignores every relevant item after that. This makes it a strong fit for search experiences where the user mainly needs one good answer quickly, but a weak fit when a task requires collecting multiple pieces of evidence.

Because of the reciprocal term, the penalty for moving the first relevant item down is steep: rank 1 gives $1.0$, rank 2 gives $0.5$, rank 5 gives $0.2$, and rank 10 gives $0.1$.

### NDCG@k
Normalized Discounted Cumulative Gain rewards relevant results near the top and discounts lower positions.

$$
DCG@k = \sum_{i=1}^{k} \frac{rel_i}{\log_2(i+1)}
$$

$$
NDCG@k = \frac{DCG@k}{IDCG@k}
$$

Theory: NDCG@k models the idea that high ranks matter more than low ranks, but unlike MRR it can reward multiple relevant results. It also supports graded relevance, so one document can be mildly relevant while another is highly relevant.

The logarithmic discount means that moving a relevant document from rank 2 to rank 8 hurts, but not as brutally as a linear penalty would. The normalization by $IDCG@k$ keeps scores in the range $[0,1]$, which makes queries with different numbers of relevant documents easier to compare.

## One shared example for MP, MRR, and NDCG

Assume the top-5 retrieved documents for one query are ranked like this:

| Rank | Document | Relevant? |
| --- | --- | --- |
| 1 | `doc_A` | 0 |
| 2 | `doc_B` | 1 |
| 3 | `doc_C` | 0 |
| 4 | `doc_D` | 1 |
| 5 | `doc_E` | 0 |

There are 2 relevant documents in the top 5, at ranks 2 and 4.

### MP@5
For this single query:

$$
Precision@5 = \frac{2}{5} = 0.4
$$

If this were the only query in the evaluation set, then:

$$
MP@5 = 0.4
$$

Interpretation: 40% of the first 5 results are useful, but this score does not tell you that the first relevant result was at rank 2 rather than rank 1.

### MRR
The first relevant result appears at rank 2, so the reciprocal rank is:

$$
\frac{1}{2} = 0.5
$$

For one query, that also means:

$$
MRR = 0.5
$$

Interpretation: the system gives the user a useful result quickly, but not immediately. The second relevant document at rank 4 does not improve MRR at all.

### NDCG@5
Using binary relevance values $rel_i \in \{0,1\}$:

$$
DCG@5 = \frac{1}{\log_2(2+1)} + \frac{1}{\log_2(4+1)}
$$

$$
DCG@5 \approx \frac{1}{1.585} + \frac{1}{2.322} \approx 0.631 + 0.431 = 1.062
$$

The ideal ranking would place both relevant documents at ranks 1 and 2:

$$
IDCG@5 = \frac{1}{\log_2(1+1)} + \frac{1}{\log_2(2+1)} = 1 + 0.631 = 1.631
$$

So:

$$
NDCG@5 = \frac{1.062}{1.631} \approx 0.651
$$

Interpretation: the ranking is decent but clearly not ideal. Unlike MRR, NDCG notices that a second relevant document is also present and gives partial credit for it.

## Same ranking, three different viewpoints
- **MP@5 = 0.4** says how clean the top-5 set is
- **MRR = 0.5** says how quickly the first useful item appears
- **NDCG@5 = 0.651** says how good the whole ranked order is relative to the best possible order

## How to choose retrieval metrics
- Use **Recall@k** when missing evidence is costly
- Use **Top-1** and **MRR** when the first result matters most
- Use **NDCG@k** when you care about the full order of relevant results
- Use **coverage / all-gold-fetched** for multi-source answers

# 8. Generation Metrics by Use Case

There is no single best generation metric. The right metric depends on the business goal.

| Use case | What success means | Best generation metrics | Notes |
| --- | --- | --- | --- |
| Exact fact lookup | Exact field, code, ID, or short fact is correct | Exact Match, token F1, groundedness | Good for schema lookup, incident IDs, product codes |
| Multi-fact QA | All required facts are present and no wrong facts appear | Required-fact recall, completeness, groundedness, judge score | Strong for analytics and business QA |
| Summarization over retrieved docs | Main ideas are preserved and supported | ROUGE, BERTScore, groundedness, citation recall | ROUGE alone is not enough |
| Policy or compliance QA | Answer is correct, cautious, and attributable | Groundedness, citation precision, citation recall, abstention accuracy | Missing citations is risky |
| Support assistant | Helpful, relevant, and safe | Answer relevance, groundedness, completeness, hallucination rate | Often needs human or judge evaluation too |
| Research / analyst copilot | Synthesis across many sources | Completeness, citation recall, source diversity, faithfulness | Retrieval coverage matters heavily |

## Core generation metrics

### Exact Match
Best when only one exact answer string is acceptable. Too strict for normal paraphrased answers.

### Token F1
Measures overlap between generated and reference tokens. Good middle ground for short factual QA.

### ROUGE and BLEU
Useful for reference overlap, but they are lexical. They do not directly measure truthfulness.

BLEU combines clipped $n$-gram precision with a brevity penalty:

$$
\operatorname{BLEU}_N = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)
$$

Where:
- $p_n$ is the modified precision for $n$-grams
- $w_n$ is the weight for each $n$-gram order, commonly $1/N$
- $BP = 1$ if candidate length $c > r$, otherwise $BP = \exp(1-r/c)$
- $c$ is candidate length and $r$ is reference length

Example with BLEU-2:
- Reference: `the system returns the correct invoice status`
- Candidate: `the system returns correct invoice status`
- Unigram precision: $p_1 = 6/6 = 1.0$
- Bigram precision: $p_2 = 4/5 = 0.8$
- Brevity penalty: $BP = \exp(1-7/6) \approx 0.846$

$$
\operatorname{BLEU}_2 = 0.846 \cdot \exp\left(\frac{1}{2}\log(1.0) + \frac{1}{2}\log(0.8)\right) \approx 0.757
$$

This rewards lexical overlap, but it still misses semantic equivalence and grounding. In RAG, BLEU is more useful for templated answers than for open-ended QA.

### Groundedness / Faithfulness
Checks whether the answer is supported by the retrieved context. This is one of the most important RAG metrics.

### Answer relevance
Checks whether the answer actually addresses the query. A grounded answer can still be irrelevant.

### Completeness
Checks whether all required facts are present. This is critical for multi-fact questions.

### Practical warning for RAG
BLEU can collapse to zero on short answers when any higher-order precision is zero. In practice, use smoothing or restrict BLEU to lower orders when you only need light lexical overlap checking.

# 9. Ground Truth Design for Hybrid RAG Evaluation

A strong evaluation dataset should separate retrieval truth from answer truth.

A practical record can look like this:

```json
{
  "id": "q_example",
  "query": "For chronic late payer reporting, which job should I inspect first?",
  "gold_sources": ["pipeline_runbook.txt", "collections_playbook.txt"],
  "expected_top_source": "pipeline_runbook.txt",
  "expected_source_order": ["pipeline_runbook.txt", "collections_playbook.txt"],
  "gold_chunk_ids": ["pipeline_runbook.txt::chunk_0", "collections_playbook.txt::chunk_0"],
  "required_answer_facts": ["job_7841", "invoice aging mart", "severe delinquency"],
  "forbidden_answer_facts": ["refund", "shipping"],
  "reference_answer": "Inspect job_7841 first because it refreshes the invoice aging mart used to identify severe delinquency."
}
```

## Why this schema is useful
- `gold_sources` measures correct fetch
- `expected_top_source` and `expected_source_order` measure ranking quality
- `required_answer_facts` and `forbidden_answer_facts` measure completeness and hallucination
- `reference_answer` supports lexical and semantic answer evaluation

# 10. Practical Evaluation Scorecard

## If the use case is knowledge lookup
Primary retrieval metrics:
- Recall@k
- Top-1 accuracy
- MRR

Primary generation metrics:
- Exact Match or token F1
- Groundedness

## If the use case is multi-source business QA
Primary retrieval metrics:
- Source Recall@k
- All-gold-fetched
- NDCG@k

Primary generation metrics:
- Required-fact recall
- Completeness
- Groundedness
- LLM judge alignment

## If the use case is regulated or auditable output
Primary retrieval metrics:
- Recall@k
- Citation recall

Primary generation metrics:
- Groundedness
- Citation precision
- Abstention accuracy
- Hallucination rate

## Recommended evaluation stack for Hybrid RAG
1. Retrieval coverage: Hit@k, Recall@k, all-gold-fetched
2. Retrieval order: Top-1, MRR, NDCG@k
3. Answer overlap: Exact Match, F1, ROUGE, BLEU when relevant
4. Answer grounding: faithfulness or judge-based groundedness
5. Answer completeness: required-fact recall and missing-fact analysis
6. Business risk: forbidden facts, hallucination rate, citation failures

## Final takeaway
A strong Hybrid RAG system is not just one with high semantic similarity. It is one that:
- fetches the right evidence
- ranks the best evidence early
- fuses sparse and dense signals well
- produces grounded and complete answers
- is measured with metrics that match the use case